# JSON, API et géocodage — Quête S09

## Mise en place : vérifier l'arborescence

In [1]:
import os

print("je m'execute depuis :", os.getcwd())
print("le dossier data existe :", os.path.isdir("../data"))
print("le fichier existe     :", os.path.isfile("../data/restaurants.json"))
print("contenu de ../data    :", os.listdir("../data"))

je m'execute depuis : C:\Users\sebas\Documents\json-api-geocodage\notebooks
le dossier data existe : True
le fichier existe     : True
contenu de ../data    : ['restaurants.json', 'restaurants_geocodes.csv', 'villes.json']


## Challenge 1 — Un JSON à deux étages
### Étape 1 — explorer sans pandas

In [2]:
import json

with open("../data/villes.json", encoding="utf-8") as f:
    villes = json.load(f)

print("nombre de villes :", len(villes["villes"]))
print("deuxième ville :", villes["villes"][1]["nom"])
print("restaurants à Syracuse :", len(villes["villes"][2]["restaurants"]))
print("premier restaurant de Palerme :", villes["villes"][1]["restaurants"][0]["nom"])

nombre de villes : 3
deuxième ville : Palerme
restaurants à Syracuse : 2
premier restaurant de Palerme : Antica Focacceria


### Étape 2 — compter à la main

In [3]:
for v in villes["villes"]:
    print(f"{v['nom']:10} {len(v['restaurants'])} restaurants")

Catane     3 restaurants
Palerme    2 restaurants
Syracuse   2 restaurants


### Étape 3 — tout mettre à plat

In [4]:
import pandas as pd

df_villes = pd.json_normalize(
    villes,
    record_path=["villes", "restaurants"],
    meta=[["villes", "nom"], ["villes", "region"], ["villes", "habitants"]],
)
df_villes = df_villes.rename(columns={"villes.nom": "ville", "villes.region": "region", "villes.habitants": "habitants"})
print(df_villes.shape)
df_villes

(7, 6)


,nom,type,notes,ville,region,habitants
0,Gelateria Zio Pietro dal 1964,glacier,"[4.6, 4.8, 4.7]",Catane,Sicile,298762
1,A Casa d'Amici,trattoria,"[4.4, 4.5]",Catane,Sicile,298762
2,La Bitta,poisson,"[4.2, 4.0, 4.3, 4.1]",Catane,Sicile,298762
3,Antica Focacceria,trattoria,[4.3],Palerme,Sicile,630828
4,Ferro di Cavallo,trattoria,[4.5],Palerme,Sicile,630828
5,Don Camillo,poisson,"[4.7, 4.8, 4.6]",Syracuse,Sicile,121605
6,Gelateria Bianca,glacier,[4.2],Syracuse,Sicile,121605


### Étape 4 — exploiter

In [5]:
df_villes["note_moyenne"] = df_villes["notes"].apply(lambda n: round(sum(n) / len(n), 2))
df_villes[["nom", "type", "ville", "region", "habitants", "note_moyenne"]]

,nom,type,ville,region,habitants,note_moyenne
0,Gelateria Zio Pietro dal 1964,glacier,Catane,Sicile,298762,4.70
1,A Casa d'Amici,trattoria,Catane,Sicile,298762,4.45
2,La Bitta,poisson,Catane,Sicile,298762,4.15
3,Antica Focacceria,trattoria,Palerme,Sicile,630828,4.30
4,Ferro di Cavallo,trattoria,Palerme,Sicile,630828,4.50
5,Don Camillo,poisson,Syracuse,Sicile,121605,4.70
6,Gelateria Bianca,glacier,Syracuse,Sicile,121605,4.20


In [6]:
print(df_villes.groupby("ville")["note_moyenne"].mean().round(2).sort_values(ascending=False))
print()
print(df_villes["type"].value_counts())

ville
Syracuse    4.45
Catane      4.43
Palerme     4.40
Name: note_moyenne, dtype: float64

type
trattoria    3
glacier      2
poisson      2
Name: count, dtype: int64


**Réponses — Challenge 1**

- **Meilleure note moyenne :** Syracuse (4,45), devant Catane (4,43) et Palerme (4,40).
- **Restaurants par type :** 3 trattorias, 2 glaciers, 2 poissons.

## Challenge 2 — Interroger une API en série

In [7]:
import requests

def interroger(url, params, headers=None):
    try:
        r = requests.get(url, params=params, headers=headers, timeout=10)
        r.raise_for_status()
        return r.json()
    except requests.exceptions.Timeout:
        print("Trop lent, on abandonne cette requête")
    except requests.exceptions.HTTPError as e:
        print("Le serveur a répondu une erreur :", e)
    except requests.exceptions.RequestException as e:
        print("Problème réseau :", e)
    return None

URL_ADRESSE = "https://api-adresse.data.gouv.fr/search/"
adresses = [
    "20 avenue de Ségur 75007 Paris",
    "Grand Place, Lille",
    "1 rue de la Paix, Paris",
    "place Bellecour, Lyon",
    "zzzzz qwerty 99999",
]

### Étape 1 — une seule, d'abord

In [8]:
r = requests.get(URL_ADRESSE, params={"q": adresses[0], "limit": 1}, timeout=10)
print(r.status_code)
props = r.json()["features"][0]["properties"]
print(props["label"], props["score"])

200
20 Avenue de Ségur 75007 Paris 0.9716281818181818


### Étape 2 — les cinq

In [9]:
resultats = []
for adresse in adresses:
    reponse = interroger(URL_ADRESSE, {"q": adresse, "limit": 1})
    features = reponse["features"] if reponse else []
    p = features[0]["properties"] if features else {}  # introuvable : 200 avec features vide
    resultats.append({
        "adresse": adresse,
        "label": p.get("label"),
        "score": p.get("score"),
        "postcode": p.get("postcode"),
        "city": p.get("city"),
    })

### Étape 3 — en DataFrame

In [10]:
df_adresses = pd.DataFrame(resultats)
df_adresses

,adresse,label,score,postcode,city
0,20 avenue de Ségur 75007 Paris,20 Avenue de Ségur 75007 Paris,0.971628,75007,Paris
1,"Grand Place, Lille",Place de Gand 59800 Lille,0.568244,59800,Lille
2,"1 rue de la Paix, Paris",1 Rue de la Paix 75002 Paris,0.964060,75002,Paris
3,"place Bellecour, Lyon",Place Bellecour 69002 Lyon,0.967367,69002,Lyon
4,zzzzz qwerty 99999,NaN,NaN,NaN,NaN


### Étape 4 — lire le résultat

In [11]:
trouvees = df_adresses.dropna(subset=["label"])
print("adresses trouvées :", len(trouvees), "sur", len(df_adresses))
print(trouvees.loc[trouvees["score"].idxmin(), ["adresse", "label", "score"]])

adresses trouvées : 4 sur 5
adresse           Grand Place, Lille
label      Place de Gand 59800 Lille
score                       0.568244
Name: 1, dtype: object


**Réponses — Challenge 2**

1. **4 adresses sur 5** ont été trouvées. `zzzzz qwerty 99999` revient vide : l'API répond 200 avec une liste `features` vide.
2. **Score le plus bas : « Grand Place, Lille » (0,57).** « Grand Place » est le nom d'usage, pas le nom officiel de la place (place du Général-de-Gaulle) : l'API a pris ce qui ressemblait le plus.
3. **Non, pas toujours.** Pour Lille, l'API a renvoyé « Place de Gand », une autre place. Les trois autres labels correspondent bien à la demande. Il faut toujours comparer le `label` et le `score` à ce qu'on a demandé.

## Challenge 3 — Les restaurants de Catane, sur une carte
### Étape 1 — du JSON au DataFrame

In [12]:
with open("../data/restaurants.json", encoding="utf-8") as f:
    data = json.load(f)

df = pd.json_normalize(data, record_path="restaurants", meta=["ville", "pays"])
df[["nom", "adresse", "type", "ville", "pays"]]

,nom,adresse,type,ville,pays
0,Gelateria Zio Pietro dal 1964,"Via Porta di Ferro, 47, 95131 Catania CT",glacier,Catane,Italie
1,A Casa d'Amici,"Via Fischetti, 14, 95131 Catania CT",trattoria,Catane,Italie
2,La Bitta,"Via Acquicella Porto, 95121 Catania CT",poisson,Catane,Italie
3,Steak House,"Via Porta di Ferro, 8, 95100 Catania CT",grill,Catane,Italie


### Étape 2 — la note moyenne

In [13]:
df["note_moyenne"] = df["notes"].apply(lambda n: round(sum(n) / len(n), 2))
df[["nom", "note_moyenne"]]

,nom,note_moyenne
0,Gelateria Zio Pietro dal 1964,4.70
1,A Casa d'Amici,4.45
2,La Bitta,4.15
3,Steak House,4.00


### Étape 3 — géocoder avec Nominatim

In [14]:
import time

def geocoder_monde(adresse):
    """Renvoie (latitude, longitude) n'importe où dans le monde, ou (None, None)."""
    url = "https://nominatim.openstreetmap.org/search"
    headers = {"User-Agent": "bootcamp-simplon-da/1.0 (projet json-api-geocodage)"}
    params = {"q": adresse, "format": "json", "limit": 1}
    try:
        r = requests.get(url, params=params, headers=headers, timeout=10)
        r.raise_for_status()
        resultats = r.json()
        if not resultats:
            return None, None
        return float(resultats[0]["lat"]), float(resultats[0]["lon"])
    except requests.exceptions.RequestException:
        return None, None
    finally:
        time.sleep(1)  # 1 requête par seconde maximum, même en cas d'échec

debut = time.time()
df[["lat", "lon"]] = df["adresse"].apply(lambda a: pd.Series(geocoder_monde(a)))
print(f"durée : {time.time() - debut:.1f} s")
df[["nom", "lat", "lon"]]

durée : 4.2 s


,nom,lat,lon
0,Gelateria Zio Pietro dal 1964,37.502093,15.093195
1,A Casa d'Amici,37.506171,15.094322
2,La Bitta,37.489440,15.079431
3,Steak House,37.502933,15.093041


### Étape 4 — la carte

In [15]:
import folium

premier = [df.loc[0, "lat"], df.loc[0, "lon"]]
# tuiles Esri : les tuiles OpenStreetMap renvoient 403 quand la carte est ouverte hors d'un site web
m = folium.Map(location=premier, zoom_start=16, tiles="Esri.WorldStreetMap")
for _, r in df.dropna(subset=["lat"]).iterrows():
    folium.Marker([r["lat"], r["lon"]], popup=f"{r['nom']} — {r['note_moyenne']}").add_to(m)
m

### Étape 5 — garder le résultat

In [16]:
df.to_csv("../data/restaurants_geocodes.csv", index=False, encoding="utf-8")
print(pd.read_csv("../data/restaurants_geocodes.csv").shape)

(4, 9)
